In [ ]:
#
import pandas as pd

In [ ]:
### cell 0 ###

filename = "/home/dias-benchmarks/new_notebooks/imdb/movie_metadata.csv"

m = pd.read_csv(filename)
factor = 500
# Build a GPU‐side index series that repeats the full block of row indices factor times
idx = pd.Series(range(len(m)))
positions = pd.concat([idx] * factor, ignore_index=True)
# Gather all the blocks in one go on the GPU
m = m.take(positions)

In [ ]:
### cell 1 ###

m.info()

In [ ]:
### cell 2 ###

# Trim whitespace on the movie titles
m.movie_title = m.movie_title.str.strip()

# Convert numeric columns to match original pd.to_numeric behavior
m = m.astype(
    {"duration": "int64", "budget": "int64", "gross": "int64", "imdb_score": "float64"}
)

# Set the movie_title as the index and drop all unwanted columns in one go
m.set_index("movie_title", inplace=True)
cols_to_drop = [
    "color",
    "director_facebook_likes",
    "actor_3_facebook_likes",
    "actor_2_name",
    "actor_1_facebook_likes",
    "actor_1_name",
    "cast_total_facebook_likes",
    "movie_imdb_link",
    "language",
    "actor_2_facebook_likes",
    "aspect_ratio",
    "actor_3_name",
    "facenumber_in_poster",
    "plot_keywords",
    "country",
]
m.drop(columns=cols_to_drop, inplace=True)

In [ ]:
### cell 3 ###

m.columns

In [ ]:
### cell 4 ###

# Compute total number of movies on the GPU
total_num_movies = m.duration.count()

# Compute fraction of movies longer than 90 min in one GPU‐side operation
ratio_long_movies = (m.duration > 90.0).mean()

ratio_long_movies

In [ ]:
### cell 5 ###

two_hour_movies = (m.duration > 120.0).sum()
two_hour_movies / total_num_movies

In [ ]:
### cell 6 ###

# Optimized for cudf
directed_mask = m.director_name.notna()
num_movies_directed = directed_mask.sum()
spiel = m.director_name.eq("Steven Spielberg").sum()
spiel / num_movies_directed

In [ ]:
### cell 7 ###

ce = m[m.director_name == "Clint Eastwood"]
under_budget = (ce.gross < ce.budget).sum()
total = len(ce)
under_budget / total

In [ ]:
### cell 8 ###

# Optimized for cudf
gross_budget_ratio = (m.gross > m.budget).dropna().mean()
# return the ratio so the cell output matches the original
gross_budget_ratio

In [ ]:
### cell 9 ###

average_gross = m.gross.mean()
movie_grossed_over_average = (m.gross > average_gross).sum()
total_movie_with_gross = m.gross.count()
movie_grossed_over_average / total_movie_with_gross

In [ ]:
### cell 10 ###

# Filter out any rows with missing imdb_score, gross, or budget
movies_with_scores = m[["imdb_score", "gross", "budget"]].dropna()

# Build boolean masks directly on the GPU DataFrame
mask_pos = movies_with_scores.imdb_score > 6
mask_profit = movies_with_scores.gross < movies_with_scores.budget

# Compute and display the false‐positive rate
(mask_pos & mask_profit).sum() / mask_pos.sum()

In [ ]:
### cell 11 ###

# Optimized using vectorized boolean masks without intermediate DataFrame slices
mask_neg = movies_with_scores.imdb_score <= 6
mask_fn = movies_with_scores.gross > movies_with_scores.budget

# Compute ratio of false negatives among negatives in one pass
default_neg_count = mask_neg.sum()
false_neg_count = (mask_neg & mask_fn).sum()
false_neg_count / default_neg_count

In [ ]:
### cell 12 ###

scores = m.imdb_score

# plt.figure(figsize=(10, 3))
# plt.hist(scores, bins=np.arange(1, 11))
# plt.title("Distribution of Ratings")
# plt.xlabel("IMDB Rating")
# plt.ylabel("# of Movies")
# plt.show()

In [ ]:
### cell 13 ###

scores.describe()

In [ ]:
### cell 14 ###

mean = scores.mean()
median = scores.quantile(0.5)
std = scores.std()
mean, median, std